In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from pydantic import SecretStr,BaseModel , Field 
from typing import TypedDict,NotRequired,Annotated, Literal
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
import operator
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver # ram wali memory

from dotenv import load_dotenv

load_dotenv()

True

In [92]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    api_key = SecretStr(os.environ["GOOGLE_API_KEY"]),
)

In [93]:
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage], add_messages]

In [94]:
def chat_node(state : ChatState) :
    # take user query from state 
    messages = state['messages']


    # send tp the llm 
    response = llm.invoke(messages)

    # response store state 
    return{
        'messages':[response]
    }

In [95]:

checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node',chat_node)

graph.add_edge(START,'chat_node')
graph.add_edge('chat_node',END)

workflow = graph.compile(checkpointer=checkpointer)

# final_state = workflow.invoke({'messages':[HumanMessage(content="what is capital of Spain?")]})
# final_state['messages'][-1].text


In [96]:
thread_id = '1'
while True:
    user_input = input("User: ")
    if user_input.lower() in ['exit', 'quit', 'bye']:
        break

    final_state = workflow.invoke({'messages': [HumanMessage(content=user_input)]}, config={'configurable': {'thread_id': thread_id}})

    for msg in final_state['messages']:
        role = 'User' if msg.type == 'human' else 'AI'
        print(f"{role}: {msg.text}")

User: what is capital of dholakpur 
AI: In the fictional universe of **Chhota Bheem**, Dholakpur itself is depicted as the capital kingdom ruled by **King Indraverma**, so it doesn't have a separate capital—it *is* the capital! 

However, in the lore of the show, Dholakpur is the central kingdom, while neighboring rival kingdoms include **Puhwan City** (ruled by Raja Indrasena) and **Mangal Singh's territory**.
User: what is capital of dholakpur 
AI: In the fictional universe of **Chhota Bheem**, Dholakpur itself is depicted as the capital kingdom ruled by **King Indraverma**, so it doesn't have a separate capital—it *is* the capital! 

However, in the lore of the show, Dholakpur is the central kingdom, while neighboring rival kingdoms include **Puhwan City** (ruled by Raja Indrasena) and **Mangal Singh's territory**.
User: what is capital of dholakhpur
AI: As I mentioned, **Dholakpur** doesn't have a capital because **Dholakpur *is* the capital city and kingdom** itself! 

In the worl

In [97]:
workflow.get_state(config={'configurable': {'thread_id': thread_id}})

StateSnapshot(values={'messages': [HumanMessage(content='what is capital of dholakpur ', additional_kwargs={}, response_metadata={}, id='19f20158-d7bd-4296-b227-d91a793914d4'), AIMessage(content=[{'type': 'text', 'text': "In the fictional universe of **Chhota Bheem**, Dholakpur itself is depicted as the capital kingdom ruled by **King Indraverma**, so it doesn't have a separate capital—it *is* the capital! \n\nHowever, in the lore of the show, Dholakpur is the central kingdom, while neighboring rival kingdoms include **Puhwan City** (ruled by Raja Indrasena) and **Mangal Singh's territory**.", 'extras': {'signature': 'El4KXAERTTIPniHRJMuN7Ic6CX1nJgbq3Bcqdnmhff7k4dDcuK2ni1B34eRX/Retk+0KdpM//tPMA7CfunJXcoMnDeBiQOjsYS3A1tpePXp/CGFdo4qA2FdN0eD5N2gA'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a07b51-afe0-7881-a4a5-f1cf299685d2-0', tool_calls=[], invalid_to